# 06 — Comparaison des modèles

**Prérequis** : notebooks 02→07 exécutés (résultats pkl générés).

**Charge** : `results_rnn.pkl`, `results_lstm.pkl`, `results_bilstm.pkl`, `results_transformer.pkl`, `results_cnn_lstm.pkl`, `results_distilbert.pkl`

**Comparaison** : test accuracy, courbes de val loss, matrices de confusion côte à côte.

## Bloc 1 : Imports & chargement

In [ ]:
import pickle, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

DATA_DIR  = '../data/'
MODEL_DIR = '../models/'

N_CLASSES = 8

with open(DATA_DIR + 'label_maps.json') as f:
    label_maps = json.load(f)
CLIENT_LABELS = {int(k): v for k, v in label_maps['client'].items()}
label_names   = [CLIENT_LABELS[i] for i in range(N_CLASSES)]

# Chargement de tous les résultats
results = {}
for name in ['rnn', 'lstm', 'bilstm', 'transformer', 'cnn_lstm', 'distilbert']:
    path = DATA_DIR + f'results_{name}.pkl'
    try:
        with open(path, 'rb') as f:
            results[name] = pickle.load(f)
        print(f'{name:12s} — test_acc={results[name]["test_accuracy"]:.4f}')
    except FileNotFoundError:
        print(f'{name:12s} — fichier non trouvé, notebook pas encore exécuté')

## Bloc 2 : Comparaison des accuracies (bar chart)

In [ ]:
available = {k: v for k, v in results.items()}
models    = [v['model'] for v in available.values()]
accs      = [v['test_accuracy'] for v in available.values()]

# Couleurs : bleu = from scratch, vert = hybride, orange = pré-entraîné
color_map = {
    'RNN'       : '#4C72B0',
    'LSTM'      : '#4C72B0',
    'BiLSTM'    : '#4C72B0',
    'Transformer': '#4C72B0',
    'CNN+BiLSTM': '#2CA02C',   # vert — architecture hybride
    'DistilBERT': '#DD8452',   # orange — pré-entraîné
}
colors = [color_map.get(m, '#4C72B0') for m in models]

plt.figure(figsize=(12, 5))
bars = plt.bar(models, accs, color=colors, edgecolor='white', linewidth=1.2)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
plt.ylim(0, 1.05)
plt.ylabel('Test Accuracy')
plt.title('Comparaison des modèles — Test Accuracy\n(bleu = from scratch | vert = hybride | orange = pré-entraîné)')
plt.axhline(max(accs), color='red', linestyle='--', alpha=0.5, label=f'Best={max(accs):.4f}')
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR + 'comparison_accuracy.png', dpi=100)
plt.show()

print('\n=== Classement ===')
for rank, (name, acc) in enumerate(sorted(zip(models, accs), key=lambda x: -x[1]), 1):
    print(f'  {rank}. {name:<15s} : {acc:.4f}')

## Bloc 3 : Courbes de val loss comparées

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, res in available.items():
    epochs = range(1, len(res['val_losses']) + 1)
    axes[0].plot(epochs, res['val_losses'], label=res['model'], marker='o', markersize=3)
    axes[1].plot(epochs, res['val_accs'],   label=res['model'], marker='o', markersize=3)

axes[0].set_title('Val Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Val Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.suptitle('Courbes entraînement — comparaison')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'comparison_curves.png', dpi=100)
plt.show()

## Bloc 4 : Matrices de confusion côte à côte

In [ ]:
n_models = len(available)
fig, axes = plt.subplots(1, n_models, figsize=(11 * n_models, 9))
if n_models == 1: axes = [axes]

for ax, (name, res) in zip(axes, available.items()):
    cm = confusion_matrix(res['targets'], res['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_title(f"{res['model']}\nacc={res['test_accuracy']:.4f}")
    ax.set_ylabel('Vrai'); ax.set_xlabel('Prédit')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.suptitle('Matrices de confusion — tous modèles', fontsize=14)
plt.tight_layout()
plt.savefig(MODEL_DIR + 'comparison_confusion.png', dpi=80)
plt.show()

## Bloc 5 : Tableau récapitulatif

In [ ]:
print('\n' + '='*65)
print(f"{'Modèle':<15} {'Test Acc':>10} {'Best Val':>10} {'Epochs':>8}")
print('='*65)
for name, res in sorted(available.items(), key=lambda x: -x[1]['test_accuracy']):
    print(f"{res['model']:<15} {res['test_accuracy']:>10.4f} {res['best_val_acc']:>10.4f} {len(res['val_accs']):>8}")
print('='*65)

print('\n=== Rapport détaillé — meilleur modèle ===')
best_name = max(available.items(), key=lambda x: x[1]['test_accuracy'])
print(f"\n{best_name[1]['model']}:")
print(best_name[1]['report'])